# RAMtools Benchmark Report

Visualizes the JSON output of the RAMtools benchmark suite. Each of the five
benchmark binaries writes one Google Benchmark JSON file into `build/benchmark/`.

Conversion time, region query and chromosome split each compare **three** backends —
TTree, RNTuple and BAM (in-process htslib); the SAM→RAM size and columnar-read sections
are ROOT-only. See `benchmark/README.md` for what each backend is timed doing, and for
the coordinate-sorting caveat that governs whether the region-query columns are
comparable at all.

## Generating the input JSON

```bash
cmake -B build -DCMAKE_BUILD_TYPE=Release
cmake --build build --target benchmark      # runs all benchmarks -> build/benchmark/*.json
```

You can also run a single binary, or a real dataset, via `scripts/run_benchmarks.sh`
(see `benchmark/README.md`). To analyze a different run, point `RESULTS_DIR` (next
cell) at the directory holding its `*.json` files.

## Running on Google Colab

The notebook is self-contained: it needs only `pandas`, `numpy` and `matplotlib`
(all preinstalled on Colab) and never touches ROOT or the build tree. Run the setup
cell with no build directory present and it prompts you to upload the benchmark
JSON files -- either the individual `*.json` or a single `.zip` of `build/benchmark/`.
You can also drag the files into Colab's `/content` file browser, or set
`RESULTS_DIR` explicitly, before running that cell.

Note that the environment table below describes the machine that *ran the
benchmarks* (read from the JSON `context`), not the Colab VM.

> These are **performance** numbers (time / size / throughput). By default they come
> from a synthetic SAM, so treat them as TTree-vs-RNTuple *comparisons*, not
> real-world absolutes. Pass `--sam=...` to the benchmarks for real data.


In [ ]:
import json
import zipfile
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

%matplotlib inline

# One JSON file per benchmark binary.
BENCH_FILES = {
    "sam_to_ram": "sam_to_ram_benchmark.json",
    "conversion": "conversion_time_benchmark.json",
    "region_query": "region_query_benchmark.json",
    "chromosome_split": "chromosome_split_benchmark.json",
    "columnar": "columnar_read_benchmark.json",
}

# --- Where the benchmark JSON files live ------------------------------------
# Set RESULTS_DIR to a path to force a specific directory; leave it None to
# auto-detect. Local runs find the build tree; on Colab (no build tree) you get
# an upload prompt that accepts the *.json files directly or a single .zip.
RESULTS_DIR = None

try:  # Colab provides google.colab.files; plain Jupyter does not.
    from google.colab import files as _colab_files
    IN_COLAB = True
except ImportError:
    _colab_files = None
    IN_COLAB = False

_UPLOAD_DIR = Path("benchmark_json")  # where uploaded/dropped files are collected


def _has_results(path):
    """True if the directory holds at least one of the expected benchmark files."""
    path = Path(path)
    return path.is_dir() and any((path / f).exists() for f in BENCH_FILES.values())


def _autodetect():
    """First candidate directory that actually contains benchmark JSON, else None."""
    candidates = [
        Path("../../build/benchmark"),  # run from benchmark/io_performance/
        Path("build/benchmark"),        # run from repo root
        Path("../build/benchmark"),     # run from benchmark/
        _UPLOAD_DIR,                    # files uploaded in a previous run of this cell
        Path("/content"),               # Colab: dragged into the file browser
    ]
    return next((p for p in candidates if _has_results(p)), None)


def _collect_upload():
    """Prompt for an upload and unpack it into _UPLOAD_DIR. Returns that directory."""
    _UPLOAD_DIR.mkdir(exist_ok=True)
    for name, blob in _colab_files.upload().items():
        dest = _UPLOAD_DIR / Path(name).name
        dest.write_bytes(blob)
        if dest.suffix == ".zip":  # a zipped build/benchmark/ works too
            with zipfile.ZipFile(dest) as zf:
                for member in zf.namelist():
                    if member.endswith(".json") and not member.endswith("/"):
                        (_UPLOAD_DIR / Path(member).name).write_bytes(zf.read(member))
            dest.unlink()
    return _UPLOAD_DIR


if RESULTS_DIR is None:
    RESULTS_DIR = _autodetect()

if RESULTS_DIR is None and IN_COLAB:
    print("No build tree found. Upload the benchmark JSON files (or a .zip of them).")
    RESULTS_DIR = _collect_upload()

if RESULTS_DIR is None:  # local, nothing built yet -- report the expected path
    RESULTS_DIR = Path("build/benchmark")

RESULTS_DIR = Path(RESULTS_DIR)

# Consistent colours for the storage backends across every plot. BAM appears in the
# conversion, region-query and chromosome-split sections; the other two are RNTuple-only.
COLORS = {"TTree": "#4C72B0", "RNTuple": "#DD8452", "BAM": "#55A868"}

# Order backends consistently wherever several are plotted together.
BACKENDS = ("TTree", "RNTuple", "BAM")

plt.rcParams["figure.figsize"] = (9, 4.5)
plt.rcParams["axes.grid"] = True
plt.rcParams["grid.alpha"] = 0.3
plt.rcParams["axes.axisbelow"] = True

print("Reading benchmark JSON from:", RESULTS_DIR.resolve())
_found = [f for f in BENCH_FILES.values() if (RESULTS_DIR / f).exists()]
print(f"Found {len(_found)}/{len(BENCH_FILES)} benchmark files:",
      ", ".join(_found) if _found else "(none -- every section below will be skipped)")


## Loading and parsing

Helpers to read the Google Benchmark JSON (`{context, benchmarks:[...]}`) into tidy
`pandas` DataFrames and to pull structured fields (backend, read count, region
index) out of the benchmark case `name`. Custom counters vary by run mode, so every
field is read defensively — a missing file just drops its section instead of raising.


In [ ]:
def load_benchmark(path):
    """Load one Google Benchmark JSON file.

    Returns (context_dict, DataFrame) with one row per benchmark case, or None if
    the file is missing or empty. Custom counters (e.g. ttree_size_mb) become columns.
    """
    path = Path(path)
    if not path.exists():
        return None
    with path.open() as fh:
        data = json.load(fh)
    rows = data.get("benchmarks", [])
    if not rows:
        return None
    return data.get("context", {}), pd.DataFrame(rows)


def load_all(results_dir):
    """Load every known benchmark file present under results_dir (missing ones skipped)."""
    results_dir = Path(results_dir)
    loaded = {}
    for key, fname in BENCH_FILES.items():
        res = load_benchmark(results_dir / fname)
        if res is not None:
            loaded[key] = res
    return loaded


def parse_backend(name):
    """Storage backend from names like 'RNTuple_Conversion/1000', 'RegionQuery/TTree/r3'
    or 'ChromosomeSplit/BAM/100000/4'. Returns None if no backend segment is present."""
    for backend in BACKENDS:
        if backend in name:
            return backend
    return None


def _trailing_ints(name):
    """Integer '/'-separated path segments, e.g. 'X/100000/2' -> [100000, 2]."""
    return [int(s) for s in name.split("/") if s.isdigit()]


def parse_reads(name):
    """Synthetic read count (first numeric path segment), or None (e.g. '.../real')."""
    ints = _trailing_ints(name)
    return ints[0] if ints else None


def reads_label(name):
    """X-axis label for a case: the read count if present, else the last name segment."""
    r = parse_reads(name)
    return str(r) if r is not None else name.split("/")[-1]


def parse_region_idx(name):
    """Region index from an 'r<idx>' segment in RegionQuery names."""
    for seg in name.split("/"):
        if seg.startswith("r") and seg[1:].isdigit():
            return int(seg[1:])
    return None


def series_colour(label):
    """Colour for a series label: an exact COLORS key, else the backend it starts with.

    Lets decorated labels such as "RNTuple (4t)" keep their backend's colour instead of
    falling through to matplotlib's default cycle (which would hand RNTuple TTree's blue).
    """
    if label in COLORS:
        return COLORS[label]
    return next((c for b, c in COLORS.items() if label.startswith(b)), None)


def grouped_bar(ax, categories, series, title="", ylabel="", xlabel=""):
    """Draw a grouped bar chart.

    categories: x tick labels; series: {label: values} aligned with categories.
    Values may contain NaN (drawn as gaps). Colours come from COLORS when known; series
    sharing a backend colour (e.g. the 2t/4t pair) are separated by fading each repeat.
    """
    n = max(len(series), 1)
    x = np.arange(len(categories))
    width = 0.8 / n
    seen = {}
    for i, (label, values) in enumerate(series.items()):
        offset = (i - (n - 1) / 2) * width
        colour = series_colour(label)
        repeat = seen.get(colour, 0)
        seen[colour] = repeat + 1
        alpha = 1.0 if colour is None else max(0.45, 1.0 - 0.35 * repeat)
        ax.bar(x + offset, values, width, label=label, color=colour, alpha=alpha)
    ax.set_xticks(x)
    ax.set_xticklabels(categories)
    ax.set_title(title)
    ax.set_ylabel(ylabel)
    if xlabel:
        ax.set_xlabel(xlabel)
    ax.legend()


loaded = load_all(RESULTS_DIR)
print("Loaded benchmarks:", ", ".join(loaded) or "(none found)")
missing = [k for k in BENCH_FILES if k not in loaded]
if missing:
    print("Missing (sections will be skipped):", ", ".join(missing))


## Environment

The context recorded in the JSON — capture this when reporting numbers so a run is
reproducible.


In [ ]:
_ctx = next((c for c, _ in loaded.values()), {})
_fields = ["date", "host_name", "num_cpus", "mhz_per_cpu",
           "cpu_scaling_enabled", "library_version", "library_build_type"]
pd.DataFrame([(f, _ctx.get(f)) for f in _fields], columns=["field", "value"])


## 1. SAM → RAM: size & compression

`sam_to_ram_benchmark` — one run per read count that writes both a TTree and an
RNTuple and records their on-disk sizes, the compression ratio (TTree / RNTuple),
and conversion throughput.


In [ ]:
res = loaded.get("sam_to_ram")
if res is None:
    print("sam_to_ram_benchmark.json not found - skipping.")
else:
    _, df = res
    df = df.assign(_reads=df["name"].map(parse_reads)).sort_values("_reads", na_position="last")
    cats = [reads_label(n) for n in df["name"]]

    fig, axes = plt.subplots(1, 3, figsize=(15, 4.2))
    grouped_bar(
        axes[0], cats,
        {"TTree": df.get("ttree_size_mb"), "RNTuple": df.get("rntuple_size_mb")},
        title="Output size", ylabel="size (MB)", xlabel="reads",
    )
    if "compression_ratio" in df:
        axes[1].bar(cats, df["compression_ratio"], color="#8172B3")
    axes[1].axhline(1.0, color="k", lw=1, ls="--")
    axes[1].set_title("Compression ratio (TTree / RNTuple)")
    axes[1].set_ylabel("ratio"); axes[1].set_xlabel("reads")
    if "reads_per_second" in df:
        axes[2].bar(cats, df["reads_per_second"], color="#937860")
    axes[2].set_title("Conversion throughput")
    axes[2].set_ylabel("reads / s"); axes[2].set_xlabel("reads")

    fig.suptitle("SAM to RAM: size & compression (sam_to_ram_benchmark)")
    fig.tight_layout()
    plt.show()


## 2. Conversion time

`conversion_time_benchmark` — SAM→TTree, SAM→RNTuple and SAM→BAM timed **separately**
across the read-count ladder, plus the resulting file size.

"Conversion" means the same thing for all three: produce a file that can answer a region
query. For the ROOT backends that is one pass with `index=true`; for BAM it is the write
plus the `.bai`, and — when the input is not already coordinate sorted, which the
synthetic generator's is not — the sort that indexing requires. The `sorted_in_loop`
counter records whether that sort was paid.

The BAM size column includes its `.bai`, since an RNTuple carries its index inside the
single `.root`. At small synthetic read counts the index dominates: a thousand reads
scattered over 26 hg19-length chromosomes make a sparse, disproportionately large `.bai`,
so read the size panel at the top of the ladder rather than the bottom.


In [ ]:
res = loaded.get("conversion")
if res is None:
    print("conversion_time_benchmark.json not found - skipping.")
else:
    _, df = res
    df = df.assign(backend=df["name"].map(parse_backend),
                   case=df["name"].map(reads_label))
    # Synthetic runs are one case per read count; a real dataset is the single case "real".
    order = sorted(df["case"].unique(), key=lambda c: int(c) if c.isdigit() else -1)

    def series(col):
        """{backend: values-by-case} for one counter, or {} if this run never emitted it."""
        if col not in df:
            return {}
        piv = df.pivot_table(index="case", columns="backend", values=col).reindex(order)
        return {b: piv[b].tolist() for b in BACKENDS if b in piv}

    fig, axes = plt.subplots(1, 3, figsize=(15, 4.2))
    grouped_bar(axes[0], order, series("real_time"),
                title="Conversion time", ylabel="time (ms)", xlabel="reads")
    # reads_per_second is only emitted by the synthetic sweep -- a real-dataset run has a
    # fixed read count, so the panel is dropped rather than left as an empty frame.
    rate = series("reads_per_second")
    if rate:
        grouped_bar(axes[1], order, rate,
                    title="Conversion throughput", ylabel="reads / s", xlabel="reads")
    else:
        axes[1].set_visible(False)
    grouped_bar(axes[2], order, series("file_size_mb"),
                title="Output file size", ylabel="size (MB)", xlabel="reads")
    fig.suptitle("Conversion time: TTree vs RNTuple vs BAM (conversion_time_benchmark)")
    fig.tight_layout()
    plt.show()


## 3. Region query

`region_query_benchmark` — region-query latency for TTree, RNTuple and BAM across a ladder
of regions (time in **seconds**). The right panel shows each backend's latency relative to
TTree: RNTuple's sparse, coarse index makes it *slower* on tiny regions (ratio > 1), which
is the documented trade-off.

All three backends reopen their file and reload their index on every call, so the columns
measure the same thing. Record counts are aligned too — the htslib iterator is filtered
with `0x904`, matching `RAMNTupleView` — so on coordinate-sorted data RNTuple and BAM
return identical counts, while TTree returns slightly more because it does no flag
filtering.

> On **unsorted** input (which is what `GenerateSAMFile()` produces) the ROOT views stop
> scanning at the first record past the region end and under-report matches, while BAM —
> necessarily sorted before it can be indexed — reports the true count. The benchmark
> prints a warning in that case; compare backends only on a coordinate-sorted dataset.


In [ ]:
res = loaded.get("region_query")
if res is None:
    print("region_query_benchmark.json not found - skipping.")
else:
    _, df = res
    df = df.assign(backend=df["name"].map(parse_backend),
                   ridx=df["name"].map(parse_region_idx))
    piv = df.pivot_table(index="ridx", columns="backend", values="real_time")
    cats = [f"r{int(i)}" for i in piv.index]

    fig, axes = plt.subplots(1, 2, figsize=(12, 4.3))
    grouped_bar(axes[0], cats,
                {b: piv[b].tolist() for b in BACKENDS if b in piv},
                title="Region-query latency", ylabel="time (s)", xlabel="region")
    if "TTree" in piv:
        ratios = {b: (piv[b] / piv["TTree"]).tolist()
                  for b in BACKENDS if b != "TTree" and b in piv}
        if ratios:
            grouped_bar(axes[1], cats, ratios, xlabel="region")
        axes[1].axhline(1.0, color="k", lw=1, ls="--")
    axes[1].set_title("Latency relative to TTree  (>1 = slower)")
    axes[1].set_ylabel("ratio"); axes[1].set_xlabel("region")
    fig.suptitle("Region query: TTree vs RNTuple vs BAM (region_query_benchmark)")
    fig.tight_layout()
    plt.show()


## 4. Chromosome split

`chromosome_split_benchmark` — producing one file per populated chromosome, RNTuple's
parallel writer (`ChromosomeSplit/RNTuple`) vs BAM's indexed extraction
(`ChromosomeSplit/BAM`), at 2 and 4 threads.

Both sides start from the **same SAM** and end with per-chromosome files, so the
comparison is end-to-end: RNTuple parses, groups, sorts each chromosome and writes in
parallel; BAM parses, sorts, writes an intermediate BAM with its `.bai`, then extracts
each chromosome through the index. That intermediate is real work a samtools-style
workflow pays, so it is inside the timed region — but it is excluded from `size_MB`,
which counts only the per-chromosome outputs on both sides.

Left: wall time by read count at each thread count. Right: total output size.

> Earlier revisions of this benchmark shelled out to `samtools view/sort/index` and named
> their rows `BM_SamtoolsSplit*`. Those rows no longer exist; the BAM path is in-process
> htslib and needs no `samtools` binary.


In [ ]:
res = loaded.get("chromosome_split")
if res is None:
    print("chromosome_split_benchmark.json not found - skipping.")
else:
    _, df = res

    def case_label(name):
        """"real" for a real-dataset run, else the synthetic read count."""
        return "real" if "/real" in name else str(int(parse_reads(name) or 0))

    df = df.assign(backend=df["name"].map(parse_backend), case=df["name"].map(case_label))
    # "threads" is a counter rather than a name segment; fall back to the second integer
    # in the name (Args({reads, threads})) if the counter is somehow absent.
    if "threads" not in df:
        df["threads"] = df["name"].map(lambda n: (_trailing_ints(n) + [np.nan])[1])

    cats = sorted(df["case"].unique(), key=lambda c: int(c) if c.isdigit() else -1)
    thread_axis = sorted(int(t) for t in df["threads"].dropna().unique())
    present = set(df["backend"].dropna())

    def by_backend_threads(col):
        """{"<backend> (<n>t)": values-by-case} for one counter."""
        out = {}
        for b in BACKENDS:
            if b not in present:
                continue
            for t in thread_axis:
                sel = df[(df.backend == b) & (df.threads == t)]
                out[f"{b} ({t}t)"] = [
                    sel[sel.case == c][col].mean() if len(sel[sel.case == c]) else np.nan
                    for c in cats
                ]
        return out

    fig, axes = plt.subplots(1, 2, figsize=(13, 4.3))
    grouped_bar(axes[0], cats, by_backend_threads("real_time"),
                title="Split wall time", ylabel="time (ms)", xlabel="reads")
    if "size_MB" in df:
        grouped_bar(axes[1], cats, by_backend_threads("size_MB"),
                    title="Per-chromosome output size", ylabel="size (MB)", xlabel="reads")
    fig.suptitle("Chromosome split: RNTuple parallel write vs BAM extraction "
                 "(chromosome_split_benchmark)")
    fig.tight_layout()
    plt.show()


## 5. Columnar read

`columnar_read_benchmark` — scanning a single column (`flag` / `mapq`) vs reading the
full record. This is RNTuple's columnar advantage: touching one field skips the rest.


In [ ]:
res = loaded.get("columnar")
if res is None:
    print("columnar_read_benchmark.json not found - skipping.")
else:
    _, df = res
    order = ["Columnar/FlagOnly", "Columnar/MapqOnly", "Columnar/FullRecord"]
    df = df.set_index("name").reindex([n for n in order if n in set(df["name"])]).reset_index()
    labels = [n.split("/")[-1] for n in df["name"]]
    colors = ["#4C72B0" if "Full" not in n else "#DD8452" for n in df["name"]]

    fig, axes = plt.subplots(1, 2, figsize=(11, 4.2))
    axes[0].bar(labels, df["real_time"], color=colors)
    axes[0].set_title("Read time"); axes[0].set_ylabel("time (ms)")
    if "items_per_second" in df:
        axes[1].bar(labels, df["items_per_second"], color=colors)
    axes[1].set_title("Throughput"); axes[1].set_ylabel("rows / s")
    fig.suptitle("Columnar read: single column vs full record (columnar_read_benchmark)")
    fig.tight_layout()
    plt.show()


## Summary table

One row per benchmark case: wall time plus the headline counter for that benchmark.


In [ ]:
# (column holding the headline metric, human unit) per benchmark
METRIC = {
    "sam_to_ram": ("compression_ratio", "TTree/RNTuple"),
    "conversion": ("file_size_mb", "MB"),
    "region_query": ("items_per_second", "rows/s"),
    "chromosome_split": ("size_MB", "MB"),
    "columnar": ("items_per_second", "rows/s"),
}
rows = []
for key, (col, unit) in METRIC.items():
    if key not in loaded:
        continue
    _, df = loaded[key]
    for _, r in df.iterrows():
        val = r[col] if col in df.columns and pd.notna(r[col]) else None
        rows.append({
            "benchmark": key,
            "case": r["name"],
            "time": round(float(r["real_time"]), 4),
            "unit": r["time_unit"],
            "metric": col,
            "value": round(float(val), 4) if val is not None else None,
            "metric_unit": unit,
        })
pd.DataFrame(rows)
